# Mapping Heart Health  
### County-Level Behavioral Risk Factors and Heart Disease Mortality

**Author:** J. Casey Brookshier  
**Date:** July 31, 2025  

**Objective:**  
Assess whether county-level behavioral risk factors—smoking, obesity, and physical inactivity—are associated with age-adjusted heart disease mortality in the U.S.

**Workflow:**  
Collect → Clean → Merge → Explore → Model → Visualize → Interpret


In [1]:
'''REPO Structure

mapping-heart-health/
├── README.md
├── data/
│   └── merged_heart_health_data.csv
├── notebooks/
│   └── heart_disease_behavioral_risk_analysis.ipynb
├── plots/
│   ├── scatter_behavioral_risk.png
│   └── correlation_heatmap.png
└── requirements.txt

'''

'Suggested Folder & File Naming for This Hypothesis\n📁 data/\n├── mortality_county_2020.csv          ← CDC WONDER/Kaggle\n├── county_health_rankings_2024.csv    ← countyhealthrankings.org\n├── rucc_codes.csv                     ← USDA (optional)\n├── census_quickfacts_county.csv       ← Manual export (optional)\n\n📁 analysis/\n├── correlation_matrix.ipynb\n├── regression_model.ipynb\n\n📁 plots/\n├── heatmap_correlations.png\n├── scatter_smoking_vs_mortality.png\n├── choropleth_mortality_counties.html\n\n📁 docs/\n├── hypothesis_notes.md\n├── policy_brief.pdf\n'

In [ ]:
## 1. Data Preparation

This section loads the finalized county-level dataset derived from:
- CDC WONDER heart disease mortality data
- County Health Rankings behavioral risk factors
- USDA Rural–Urban Continuum Codes

Data were cleaned and merged using FIPS codes prior to analysis.


In [ ]:
import pandas as pd
import numpy as np

# Relative path for GitHub
DATA_PATH = "../data/merged_heart_health_data.csv"

df = pd.read_csv(DATA_PATH, dtype={"fips": str})

# Keep only relevant columns for analysis
df = df[
    [
        "fips",
        "mortality_rate",
        "smoking_rate",
        "obesity_rate",
        "inactivity_rate",
        "rucc_code"
    ]
].dropna()

df.shape


In [ ]:
## 2. Exploratory Analysis

We examine pairwise relationships between behavioral risk factors and
age-adjusted heart disease mortality using:
- Pearson correlations
- Scatter plots with fitted regression lines


In [ ]:
from scipy.stats import pearsonr

risk_factors = ["smoking_rate", "obesity_rate", "inactivity_rate"]

for var in risk_factors:
    corr, p = pearsonr(df[var], df["mortality_rate"])
    print(f"{var}: r = {corr:.3f}, p < 0.001")


In [ ]:
# Visualization

import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, var in zip(axes, risk_factors):
    sns.regplot(
        data=df,
        x=var,
        y="mortality_rate",
        scatter_kws={"alpha": 0.25},
        line_kws={"color": "red"},
        ax=ax
    )
    ax.set_title(f"{var.replace('_', ' ').title()} vs Mortality")

plt.tight_layout()
plt.savefig("../plots/scatter_behavioral_risk.png", dpi=300)
plt.show()


In [ ]:
# Heatmap

corr_matrix = df[
    ["mortality_rate"] + risk_factors
].corr()

plt.figure(figsize=(5, 4))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")

plt.tight_layout()
plt.savefig("../plots/correlation_heatmap.png", dpi=300)
plt.show()


In [ ]:
## 3. Regression Modeling

Ordinary Least Squares (OLS) regression is used to estimate the association
between behavioral risk factors and heart disease mortality.

Both univariate and multivariate models are evaluated.


In [ ]:
import statsmodels.api as sm

X = df[risk_factors]
y = df["mortality_rate"]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
model.summary()


In [ ]:
## Key Findings

- Smoking, obesity, and physical inactivity are all **positively and significantly**
  associated with heart disease mortality at the county level.
- Physical inactivity and smoking show the largest effect sizes.
- The multivariate model explains approximately **17% of the variation** in mortality.

## Limitations

- Ecological (county-level) analysis
- Spatial autocorrelation not explicitly modeled
- Omitted social, environmental, and healthcare access factors

## Conclusion

Behavioral risk factors are important contributors to geographic disparities
in heart disease mortality, but they represent only part of a broader
structural and social context influencing cardiovascular outcomes.
